# Deterministic Defect Retrieval

Load synthetic manufacturing controls and retrieve records using exact filters. This notebook uses only the Python standard library: no RAG, index, API key or LLM.


In [2]:
from pathlib import Path
import sys

# Support Jupyter launched from the repository root or notebooks/.
repo_root = Path.cwd()
if not (repo_root / "est_prog" / "capstone_reporting").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from est_prog.capstone_reporting import load_defects, filter_defects


## 1. Load the dataset

`load_defects()` reads `data/synthetic_defects.json` using a path relative to the module, independently of the working directory. You can also pass an explicit JSON path. The result contains `dataset_id` and `controls`; each control retains its original nested defect records.


In [3]:
dataset = load_defects()
print(dataset["dataset_id"])
print(f"Loaded {len(dataset["controls"])} controls")
dataset["controls"][0]


synthetic-defects-v1
Loaded 6 controls


{'control_id': 'DEMO-CTRL-001',
 'job_id': 'DEMO-JOB-A',
 'section': 'TE',
 'operation': 'DEMO-TEST',
 'date': '2026-09-01',
 'sample_quantity': 100,
 'defects': []}

## 2. Retrieve relevant records

Filters are optional and combined with AND. Section matches exactly; date limits are inclusive ISO calendar dates (`YYYY-MM-DD`). Source order and controls without defects are preserved. No matches returns an empty list; invalid filters raise `ValueError`.


In [4]:
records = filter_defects(
    dataset,
    section="TE",
    date_from="2026-09-01",
    date_to="2026-09-30",
)
records


[{'control_id': 'DEMO-CTRL-001',
  'job_id': 'DEMO-JOB-A',
  'section': 'TE',
  'operation': 'DEMO-TEST',
  'date': '2026-09-01',
  'sample_quantity': 100,
  'defects': []},
 {'control_id': 'DEMO-CTRL-002',
  'job_id': 'DEMO-JOB-B',
  'section': 'TE',
  'operation': 'DEMO-TEST',
  'date': '2026-09-15',
  'sample_quantity': 80,
  'defects': [{'record_id': 'DEMO-DEF-001',
    'defect_code': 'DEMO-CODE-A',
    'errors': 8,
    'affected_boards': 6,
    'repaired_boards': 2,
    'removed_boards': 3},
   {'record_id': 'DEMO-DEF-002',
    'defect_code': 'DEMO-CODE-B',
    'errors': 5,
    'affected_boards': 4,
    'repaired_boards': 1,
    'removed_boards': 1}]},
 {'control_id': 'DEMO-CTRL-004',
  'job_id': 'DEMO-JOB-C',
  'section': 'TE',
  'operation': 'DEMO-TEST',
  'date': '2026-09-30',
  'sample_quantity': 40,
  'defects': [{'record_id': 'DEMO-DEF-004',
    'defect_code': 'DEMO-CODE-C',
    'errors': 10,
    'affected_boards': 9,
    'repaired_boards': 1,
    'removed_boards': 8}]}]

## 3. Verify known results

The fixed synthetic fixture makes these expected results reproducible. Retrieval returns original records and performs no manufacturing calculations.


In [6]:
assert [row["control_id"] for row in records] == [
    "DEMO-CTRL-001", "DEMO-CTRL-002", "DEMO-CTRL-004"
]
assert records[0]["defects"] == []
assert filter_defects(dataset, section="UNKNOWN") == []
assert filter_defects(dataset) == dataset["controls"]
print("Verified: 3 TE controls in September, including the control without defects.")


Verified: 3 TE controls in September, including the control without defects.
